# TF_IDF Comparison

# Semantic Retrieval Comparison: MiniLM, SPECTER, SciBERT, MPNet
# CLEF 2025 - CheckThat! Lab - Subtask 4b


In [1]:
!pip install sentence-transformers

  Using cached sentence_transformers-4.1.0-py3-none-any.whl.metadata (13 kB)
Using cached sentence_transformers-4.1.0-py3-none-any.whl (345 kB)


In [2]:

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
import torch
from tqdm import tqdm

tqdm.pandas()



# Load datasets (adjust paths if needed)


In [3]:
df_query_dev = pd.read_csv('subtask4b_query_tweets_dev.tsv', sep='\t')
df_collection = pd.read_pickle('subtask4b_collection_data.pkl')



# Prepare document corpus and IDs


In [4]:
docs = (df_collection['title'].fillna('') + ' ' + df_collection['abstract'].fillna('')).tolist()
doc_ids = df_collection['cord_uid'].tolist()



# Prepare queries and ground-truth document IDs


In [5]:
queries = df_query_dev['tweet_text'].fillna('').tolist()
gold_doc_ids = df_query_dev['cord_uid'].tolist()



# Helper: Evaluate MRR@k


In [6]:
def get_performance_mrr(data, col_gold, col_pred, list_k=[1, 5]):
    d_performance = {}
    for k in list_k:
        data["in_topx"] = data.apply(
            lambda x: (1 / ([i for i in x[col_pred][:k]].index(x[col_gold]) + 1)
                       if x[col_gold] in [i for i in x[col_pred][:k]] else 0),
            axis=1
        )
        d_performance[k] = data["in_topx"].mean()
    return d_performance



# Function to compute top-k document retrieval using a transformer model


In [7]:
def run_model(model_name, column_name, top_k=5):
    print(f"\nRunning model: {model_name}")
    model = SentenceTransformer(model_name)
    
    print("Encoding documents...")
    doc_embeddings = model.encode(docs, convert_to_tensor=True, show_progress_bar=True)

    print("Encoding queries...")
    query_embeddings = model.encode(queries, convert_to_tensor=True, show_progress_bar=True)

    print("Retrieving top-k documents...")
    topk_results = []
    for query_embedding in tqdm(query_embeddings):
        cos_scores = util.cos_sim(query_embedding, doc_embeddings)[0]
        top_results = torch.topk(cos_scores, k=top_k)
        indices = top_results.indices.cpu().numpy()
        top_doc_ids = [doc_ids[i] for i in indices]
        topk_results.append(top_doc_ids)

    df_query_dev[column_name] = topk_results
    result = get_performance_mrr(df_query_dev, 'cord_uid', column_name, list_k=[1, 5])
    print(f"{model_name} MRR@1: {result[1]:.4f}, MRR@5: {result[5]:.4f}")
    return result



In [9]:
results_all = {}


print("Running model: all-MiniLM-L6-v2")
results_all['MiniLM'] = run_model('all-MiniLM-L6-v2', 'minilm_topk')

Running model: all-MiniLM-L6-v2

Running model: all-MiniLM-L6-v2
Encoding documents...


Batches:   0%|          | 0/242 [00:00<?, ?it/s]

Encoding queries...


Batches:   0%|          | 0/44 [00:00<?, ?it/s]

Retrieving top-k documents...


100%|██████████| 1400/1400 [00:04<00:00, 328.20it/s]

all-MiniLM-L6-v2 MRR@1: 0.4157, MRR@5: 0.4897


In [10]:

print("\nMiniLM Results:")
print(f"MRR@5 = {results_all['MiniLM'][5]:.4f}")


MiniLM Results:
MRR@5 = 0.4897


In [11]:
print("Running model: all-mpnet-base-v2")
results_all['MPNet'] = run_model('all-mpnet-base-v2', 'mpnet_topk')

print("\nMPNet Results:")
print(f"MRR@5 = {results_all['MPNet'][5]:.4f}")

Running model: all-mpnet-base-v2

Running model: all-mpnet-base-v2
Encoding documents...


Batches:   0%|          | 0/242 [00:00<?, ?it/s]

Encoding queries...


Batches:   0%|          | 0/44 [00:00<?, ?it/s]

Retrieving top-k documents...


100%|██████████| 1400/1400 [00:19<00:00, 70.99it/s]


all-mpnet-base-v2 MRR@1: 0.4300, MRR@5: 0.5052

MPNet Results:
MRR@5 = 0.5052
